# ViAmpleHate on ViHSD: PhoBERT Proposed

This notebook implements **ViAmpleHate** — an improved version of AmpleHate (Lee et al., EMNLP 2025)
adapted for Vietnamese hate speech on the **ViHSD** dataset using **PhoBERT** as the encoder.

Improvements over the baseline (see `docs/improvementAmpleHate.md`):
1. **Vietnamese NER** (`NlpHUST/ner-vietnamese-electra-base`) replaces English CoNLL-2003 NER
2. **Vietnamese hate-target lexicon** complements NER for non-entity targets
3. **Word segmentation alignment**: NER runs on raw text, mapped to segmented token positions
4. **ContrastiveLossCosine** enabled as auxiliary training signal (CE + λ·CL)

Not applied (see optional cells):
- Improvement 6 (e-sweep): commented cell after training
- Improvement 7 (max_length): truncation profiling cell included; MAX_LEN=128 by default
- Improvement 8 (PhoBERT-large): config flag USE_PHOBERT_LARGE=False

In [ ]:
!pip install transformers datasets sentencepiece huggingface_hub underthesea easydict -q

In [ ]:
import os, re, time, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report, confusion_matrix
)
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup, pipeline

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PIN_MEMORY = DEVICE.type == 'cuda'
NUM_WORKERS = 2 if os.cpu_count() and os.cpu_count() > 2 else 0

print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Workers: {NUM_WORKERS} | Pin memory: {PIN_MEMORY}')

## 2. Hyperparameters

Key differences from original AmpleHate:
- `MODEL_NAME`: PhoBERT instead of BERT-base-uncased
- `MAX_LEN`: 128 (vs 512 in original) for Kaggle T4 efficiency
- `e`: 1.0 (middle of original search range [0.5, 1.5])
- `WARMUP_RATIO`: added for stability (not in original, but standard practice)
- `NUM_CLASSES`: 2 (NON-HATE=0, HATE=1) — same as original binary setting

In [ ]:
# Encoder
MODEL_NAME   = 'vinai/phobert-base'
NER_MODEL    = 'NlpHUST/ner-vietnamese-electra-base'  # Improvement 1: Vietnamese NER
MAX_LEN      = 128   # see truncation profiling cell below; increase to 256 if >5% truncated
HIDDEN_DIM   = 768
HEAD_DIM     = HIDDEN_DIM

# Improvement 8: PhoBERT-large option (disabled by default — doubles VRAM, needs BATCH_SIZE=8)
USE_PHOBERT_LARGE = False
if USE_PHOBERT_LARGE:
    MODEL_NAME = 'vinai/phobert-large'
    HIDDEN_DIM = 1024
    HEAD_DIM   = 1024

# AmpleHate injection strength (e in original paper)
E_INJECTION  = 1.0   # Improvement 6: tune [0.5, 0.75, 1.0, 1.25, 1.5] — see commented sweep cell

# Improvement 5: ContrastiveLoss weights
LAMBDA_CL          = 0.1    # auxiliary contrastive loss weight; tune [0.05, 0.1, 0.2]
CONTRASTIVE_MARGIN = 0.5

# Training
BATCH_SIZE   = 16    # reduce to 8 if USE_PHOBERT_LARGE=True
NUM_EPOCHS   = 6
LR           = 2e-5
HEAD_LR      = 5e-5
WARMUP_RATIO = 0.06
DROPOUT      = 0.1
PATIENCE     = 2
WEIGHT_DECAY    = 0.01
LABEL_SMOOTHING = 0.05

# Labels
NUM_CLASSES  = 2
LABEL_NAMES  = ['NON-HATE', 'HATE']

# Checkpointing
CKPT_NAME    = 'best_viamplehate_phobert_vihsd.pt'
PLOT_TITLE   = 'ViAmpleHate (PhoBERT) — ViHSD Proposed'

In [ ]:
# Improvement 2: Vietnamese hate-target lexicon
# Covers targets that NER misses: derogatory pronouns, gender, LGBTQ+, regional,
# ethnic, religious, political, occupation, social class, age, appearance groups.
VIET_TARGET_LEXICON = {

    # ── Derogatory pronouns / group markers ──────────────────────
    "thằng", "bọn", "tụi", "đứa", "mấy đứa",
    "lũ", "đám", "cái loại", "hạng", "loại người",
    "chúng mày", "chúng nó", "bọn chúng", "mấy thằng",
    "cái thứ", "đồ", "quân",

    # ── Gender ────────────────────────────────────────────────────
    "đàn ông", "đàn bà", "phụ nữ", "con gái", "con trai",
    "người phụ nữ", "người đàn ông", "giới nữ", "giới nam",
    "đàn bà con gái", "đàn ông con trai",
    "phụ nữ lái xe", "đàn bà mồm",

    # ── Sexual orientation / gender identity ──────────────────────
    "lgbt", "lgbtq", "đồng tính", "đồng tính luyến ái",
    "gay", "les", "lesbian", "bisexual", "bi",
    "chuyển giới", "transgender", "phi nhị giới",
    "bê đê", "pê đê", "bóng", "bóng lộ",
    "ái nam ái nữ", "lưỡng tính",

    # ── Regional / geographic ─────────────────────────────────────
    "người bắc", "dân bắc", "người miền bắc", "bắc kỳ",
    "người nam", "người miền nam", "dân nam kỳ", "nam kỳ",
    "người miền trung", "dân miền trung", "trung kỳ",
    "người hà nội", "người sài gòn", "người hồ chí minh",
    "dân tỉnh lẻ", "người quê", "dân quê", "nhà quê",
    "dân ngoại tỉnh", "người ngoại tỉnh",
    "dân thành thị", "dân nông thôn",

    # ── Ethnicity / nationality ───────────────────────────────────
    "người kinh", "người thượng", "người dân tộc",
    "dân tộc thiểu số", "người thiểu số",
    "người tàu", "người trung quốc", "người hoa", "hoa kiều",
    "người chăm", "người khmer", "người mường",
    "người tày", "người nùng", "người hmong", "người mông",
    "người việt", "việt nam",
    "người nước ngoài", "tây", "tây ba lô",
    "việt kiều", "người việt hải ngoại", "người mỹ gốc việt",
    "người hàn", "người nhật", "người thái",
    "chệt", "chệt hoa",

    # ── Religion / creed ─────────────────────────────────────────
    "hồi giáo", "đạo hồi", "muslim", "người hồi giáo",
    "thiên chúa giáo", "công giáo", "đạo thiên chúa",
    "tin lành", "đạo tin lành",
    "phật giáo", "đạo phật", "người theo phật",
    "cao đài", "hòa hảo",
    "người theo đạo", "con chiên", "tín đồ",
    "vô thần", "người vô thần",

    # ── Politics / ideology ───────────────────────────────────────
    "đảng viên", "đảng cộng sản", "cộng sản",
    "chế độ", "nhà nước", "chính quyền",
    "phản động", "việt cộng", "thế lực thù địch",
    "dân chủ", "đối lập", "nhân quyền",
    "thân cộng", "chống cộng",
    "tư bản", "xã hội chủ nghĩa",

    # ── Occupation ────────────────────────────────────────────────
    "công an", "cảnh sát", "cảnh sát giao thông",
    "bộ đội", "quân đội", "chiến sĩ",
    "cán bộ", "quan chức", "lãnh đạo", "chính trị gia",
    "đại biểu", "nghị sĩ",
    "nhà báo", "phóng viên", "báo chí",
    "giáo viên", "thầy giáo", "cô giáo", "giảng viên",
    "bác sĩ", "y tá", "y bác sĩ", "nhân viên y tế",
    "luật sư", "thẩm phán",
    "youtuber", "tiktoker", "streamer", "influencer",
    "kol", "idol",

    # ── Social class / economic status ───────────────────────────
    "người nghèo", "dân nghèo", "hộ nghèo",
    "người giàu", "nhà giàu", "trọc phú", "đại gia",
    "tầng lớp trung lưu", "dân lao động",
    "công nhân", "nông dân", "người lao động",
    "ăn mày", "vô gia cư", "người vô gia cư",

    # ── Age ───────────────────────────────────────────────────────
    "người già", "ông già", "bà già", "lão",
    "cụ già", "người cao tuổi",
    "giới trẻ", "thanh niên", "lũ trẻ", "bọn nhóc",
    "thế hệ z", "gen z", "gen y", "millennials",
    "trẻ trâu",

    # ── Appearance / body / disability ───────────────────────────
    "người béo", "người mập", "đồ béo",
    "người gầy", "que củi",
    "người lùn", "người cao",
    "người xấu", "người đẹp",
    "người khuyết tật", "người tàn tật",
    "người điếc", "người mù", "người câm",
    "người tâm thần", "người điên",

    # ── Mental health ─────────────────────────────────────────────
    "người trầm cảm", "người lo âu", "bệnh tâm lý",
    "bệnh tâm thần",

    # ── Immigration / social status ───────────────────────────────
    "người nhập cư", "dân nhập cư", "người di cư",
    "người tị nạn",

    # ── Implicit / indirect reference patterns ────────────────────
    "tất cả", "toàn bộ", "hết thảy",
    "đặc trưng", "bản chất", "nòi",
    "giống nòi", "dòng giống",
}

print(f"VIET_TARGET_LEXICON: {len(VIET_TARGET_LEXICON)} terms loaded")

## 3. Load ViHSD Dataset

Loading from HuggingFace Hub. Requires a Kaggle secret `HF_TOKEN`.
Original ViHSD has 3 labels: CLEAN=0, OFFENSIVE=1, HATE=2.
We remap to binary: NON-HATE=0 (CLEAN+OFFENSIVE), HATE=1 (HATE only).
This matches the mapping used in the PhoBERT-CNN baseline.

In [ ]:
from kaggle_secrets import UserSecretsClient
from datasets import load_dataset
import huggingface_hub

secret_value = UserSecretsClient().get_secret("HF_TOKEN")
huggingface_hub.login(token=secret_value, add_to_git_credential=False)

ds       = load_dataset("sonlam1102/vihsd")
train_df = ds["train"].to_pandas()
val_df   = ds["validation"].to_pandas()
test_df  = ds["test"].to_pandas()

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"Columns: {train_df.columns.tolist()}")
print(f"Label distribution (raw): {train_df['label_id'].value_counts().to_dict()}")


## 4. Label Mapping

ViHSD original: CLEAN=0, OFFENSIVE=1, HATE=2  
Binary mapping: NON-HATE=0 (CLEAN+OFFENSIVE merged), HATE=1 (HATE only)

In [ ]:
train_df['label_id'] = train_df['label_id'].map(lambda x: 1 if x == 2 else 0)
val_df['label_id']   = val_df['label_id'].map(lambda x: 1 if x == 2 else 0)
test_df['label_id']  = test_df['label_id'].map(lambda x: 1 if x == 2 else 0)

label_map = {0: 'NON-HATE', 1: 'HATE'}
for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    dist = df['label_id'].value_counts().sort_index().rename(label_map)
    print(f"  {name}: {dist.to_dict()}")


## 5. Vietnamese Text Preprocessing

Standard Vietnamese NLP preprocessing: teencode normalization, URL/email/phone removal,
repeated character collapsing, underthesea word tokenization.
Reused from the PhoBERT-CNN baseline notebook unchanged.

In [ ]:
from underthesea import word_tokenize

TEENCODE_MAP = {
    'ko': 'không', 'kh': 'không', 'khong': 'không', 'kg': 'không',
    'hok': 'không', 'hk': 'không', 'hem': 'không', 'kô': 'không',
    'chx': 'chưa', 'chua': 'chưa',
    'r': 'rồi', 'rui': 'rồi', 'ròi': 'rồi', 'oy': 'rồi', 'uj': 'rồi',
    'mk': 'mình', 'mik': 'mình', 'mh': 'mình',
    'tui': 'tôi', 'tau': 'tao',
    'may': 'mày', 'mi': 'mày',
    'bn': 'bạn', 'ban': 'bạn',
    'no': 'nó', 'mng': 'mọi người', 'mn': 'mọi người', 'ae': 'anh em',
    'dc': 'được', 'đc': 'được', 'dk': 'được', 'đk': 'được',
    'đươc': 'được', 'duoc': 'được',
    'vs': 'với', 'voi': 'với',
    'j': 'gì', 'zì': 'gì', 'zi': 'gì',
    'ntn': 'như thế nào', 'nso': 'như sao',
    'biet': 'biết', 'bit': 'biết', 'hieu': 'hiểu', 'nghi': 'nghĩ',
    'muon': 'muốn', 'hoac': 'hoặc', 'neu': 'nếu', 'nen': 'nên',
    'giet': 'giết', 'chui': 'chửi', 'danh': 'đánh',
    'nx': 'nhưng', 'nhg': 'nhưng', 'nhưg': 'nhưng', 'nma': 'nhưng mà',
    'cx': 'cũng', 'cg': 'cũng', 'cung': 'cũng', 'cũg': 'cũng',
    'ms': 'mới', 'boi': 'bởi',
    'oke': 'ok', 'okie': 'ok', 'okê': 'ok', 'okey': 'ok',
    'uh': 'ừ', 'uk': 'ừ', 'uhm': 'ừ',
    'yep': 'đúng', 'yup': 'đúng',
    'haha': 'haha', 'hehe': 'hehe', 'hihi': 'hehe', 'huhu': 'buồn',
    'haiz': 'thở dài', 'haizz': 'thở dài',
    'wtf': 'cái gì vậy', 'omg': 'ôi trời',
    'lol': 'buồn cười', 'lmao': 'buồn cười',
    'fck': 'chửi thề', 'fk': 'chửi thề', 'gg': 'xong rồi', 'ez': 'dễ',
    'bt': 'bình thường', 'bth': 'bình thường',
    'noob': 'tệ', 'nub': 'tệ',
    'xàm': 'vô nghĩa', 'nhảm': 'vô nghĩa',
    'pro': 'giỏi',
    'vl': 'vãi lồn', 'vcl': 'vãi cái lồn', 'vkl': 'vãi kép lồn',
    'vll': 'vãi lồn', 'vleu': 'vãi lồn', 'vloz': 'vãi lồn',
    'dm': 'đụ má', 'đm': 'đụ má', 'd.m': 'đụ má', 'đ.m': 'đụ má',
    'đmm': 'đụ má mày', 'dmm': 'đụ má mày',
    'đtm': 'địt mẹ', 'dtm': 'địt mẹ',
    'cl': 'cái lồn', 'lon': 'lồn', 'loz': 'lồn', 'l0n': 'lồn',
    'đéo': 'không', 'deo': 'không', 'éo': 'không',
    'cc': 'cái con', 'thg': 'thằng',
    'ngu': 'ngu', 'đần': 'đần độn', 'khùng': 'điên', 'dien': 'điên',
    'cút': 'cút', 'cut': 'cút',
    'câm': 'câm miệng', 'im mồm': 'câm miệng',
    'fb': 'facebook', 'yt': 'youtube', 'tt': 'tiktok', 'zl': 'zalo',
    'ig': 'instagram', 'cmt': 'bình luận', 'rep': 'trả lời',
    'vn': 'việt nam', 'hn': 'hà nội', 'hcm': 'hồ chí minh', 'sg': 'sài gòn',
    'iu': 'yêu', 'ieu': 'yêu',
}

def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\b0[0-9]{9,10}\b', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = [TEENCODE_MAP.get(w, w) for w in text.split()]
    return ' '.join(words)

def preprocess(text: str) -> str:
    text = normalize_text(text)
    if not text:
        return ''
    try:
        return word_tokenize(text, format='text')
    except Exception:
        return text

sample = 'mày ko hiểu gì hết, ngu vcl!!'
print(f'Original : {sample}')
print(f'Processed: {preprocess(sample)}')


## 6. Apply Preprocessing

In [ ]:
print("Preprocessing texts...")
for df, name in [(train_df, 'Train'), (val_df, 'Val'), (test_df, 'Test')]:
    df['raw_text']       = df['free_text'].apply(normalize_text)   # unsegmented — for NER (Improvement 3)
    df['text_processed'] = df['free_text'].apply(preprocess)       # word-segmented — for PhoBERT
    print(f'  {name}: done')

print("\nSample:")
for _, row in train_df.sample(3, random_state=42).iterrows():
    lbl = label_map[row['label_id']]
    print(f'  [{lbl:>8}] {row["free_text"][:50]}')
    print(f'  raw      -> {row["raw_text"][:50]}')
    print(f'  seg      -> {row["text_processed"][:50]}')

## 7. PhoBERT Tokenizer

PhoBERT uses a RoBERTa-style BPE tokenizer. Word-segmented Vietnamese text
(from underthesea) maps well to PhoBERT's vocabulary.

In [ ]:
print('Loading PhoBERT tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Vocab size: {tokenizer.vocab_size:,}')

sample = 'mày ko hiểu gì hết, ngu vcl!!'
print(f'\nOriginal : {sample}')
print(f'Processed: {preprocess(sample)}')
print(f'Tokens   : {tokenizer.tokenize(preprocess(sample))}')

In [ ]:
# Improvement 7: Truncation profiling
# If >5% of training samples are truncated, consider MAX_LEN=256 (and BATCH_SIZE=8 for VRAM).
tokenized_lengths = [
    len(tokenizer.tokenize(t)) for t in train_df['text_processed']
]
series = pd.Series(tokenized_lengths)
print("Token length distribution (training set):")
print(series.describe().round(1))
n_truncated = sum(l > MAX_LEN - 2 for l in tokenized_lengths)
pct = n_truncated / len(tokenized_lengths) * 100
print(f"\nTruncated at MAX_LEN={MAX_LEN}: {n_truncated:,} / {len(tokenized_lengths):,} ({pct:.1f}%)")
if pct > 5:
    print("WARNING: >5% truncated. Consider setting MAX_LEN=256 and BATCH_SIZE=8.")
else:
    print("OK: truncation rate is acceptable.")

## 8. ViAmpleHate Target Identification: Vietnamese NER + Lexicon

**Improvement 1:** Uses `NlpHUST/ner-vietnamese-electra-base` (ELECTRA fine-tuned on VLSP NER).
Entity types filtered: `PER`, `ORG`, `LOC`, `MISC` — the VLSP types that correspond to hate targets
(persons/groups, organizations, locations, miscellaneous groups).

**Improvement 2:** `VIET_TARGET_LEXICON` supplements NER for targets that are common nouns
(e.g., "thằng", "bọn", gender terms, regional slurs) which NER cannot capture.

**Improvement 3:** NER runs on **unsegmented** raw text to avoid PhoBERT word-segmentation
mismatch. Entity surface forms are then mapped to segmented token positions via
`ht.replace(' ', '_')` alignment with underthesea compound-word output.

Expected NER coverage improvement: ~0.09% (baseline) → 20–40% (Vietnamese NER + lexicon).

In [ ]:
class NERTagger:
    """Vietnamese NER tagger using NlpHUST/ner-vietnamese-electra-base (Improvement 1).
    Filters VLSP entity types: PER, ORG, LOC, MISC.
    """
    def __init__(self, model_name=NER_MODEL):
        self.ner_pipeline = pipeline(
            "ner",
            model=model_name,
            aggregation_strategy="simple",
            device=0 if DEVICE.type == 'cuda' else 'cpu'
        )

    def extract_named_entities(self, text):
        entities = self.ner_pipeline(text)
        # VLSP Vietnamese NER types aligned with hate targets:
        # PER=person/group, ORG=organization, LOC=location, MISC=miscellaneous groups
        target_types = {"PER", "ORG", "LOC", "MISC"}
        return [
            e["word"] for e in entities
            if e["entity_group"] in target_types
        ]


class NERProcessor:
    """Tokenizes text and finds head_token_idx using Vietnamese NER + lexicon (Improvements 2, 3).

    tokenize_and_encode(text_segmented, text_raw):
      - Runs NER on text_raw (unsegmented) to avoid segmentation mismatch
      - Scans VIET_TARGET_LEXICON on text_segmented (lowercased)
      - Maps found terms to PhoBERT token positions via underscore-joined alignment
      - Falls back to [0] (CLS) when no targets found (original AmpleHate behavior)
    """
    def __init__(self, tokenizer, ner_tagger=None, use_ner=True):
        self.tokenizer = tokenizer
        self.ner_tagger = ner_tagger
        self.use_ner = use_ner

    def extract_head_tokens(self, text_raw, text_segmented):
        tokens_found = []
        # 1. Vietnamese NER on unsegmented text (Improvement 3)
        if self.use_ner and self.ner_tagger is not None:
            tokens_found.extend(self.ner_tagger.extract_named_entities(text_raw))
        # 2. Lexicon scan on segmented text (Improvement 2)
        text_lower = text_segmented.lower()
        for term in VIET_TARGET_LEXICON:
            if term in text_lower:
                tokens_found.append(term)
        return tokens_found

    def tokenize_and_encode(self, text_segmented, text_raw=""):
        head_tokens = self.extract_head_tokens(text_raw, text_segmented)

        encoding = self.tokenizer(
            text_segmented,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN
        )
        token_ids = encoding["input_ids"]
        attention_mask = encoding["attention_mask"]

        # Map entity surface forms to segmented token positions (Improvement 3)
        seg_tokens = self.tokenizer.tokenize(text_segmented)
        head_token_idx = []
        for ht in head_tokens:
            ht_seg = ht.replace(' ', '_')  # align to underthesea compound-word format
            for i, tok in enumerate(seg_tokens):
                if ht_seg in tok or tok.replace('▁', '') == ht_seg:
                    idx = i + 1  # +1 for [CLS] at position 0
                    if idx < MAX_LEN - 1:
                        head_token_idx.append(idx)
                        break

        if not head_token_idx:
            head_token_idx = [0]  # CLS fallback (original AmpleHate behavior)

        return token_ids, head_token_idx, attention_mask

### Load English NER Model

In [ ]:
print("Loading Vietnamese NER model (Improvement 1)...")
print(f"Model: {NER_MODEL}")
ner_tagger = NERTagger()
ner_processor_train = NERProcessor(tokenizer, ner_tagger=ner_tagger, use_ner=True)
ner_processor_eval  = NERProcessor(tokenizer, ner_tagger=None, use_ner=False)

# Verify Vietnamese NER + lexicon coverage
test_texts = [
    "thằng đó là người Hà Nội",           # regional target — lexicon hit
    "Tao ghét tụi LGBT",                   # LGBTQ+ target — lexicon hit
    "người Bắc toàn nói xàm",             # regional target — lexicon hit
    "cộng sản tham nhũng hết",             # political target — lexicon hit
    "Hôm nay trời đẹp quá",               # no target
]
print("\nVietnamese NER + lexicon check:")
for t in test_texts:
    raw = normalize_text(t)
    seg = preprocess(t)
    entities = ner_tagger.extract_named_entities(raw)
    lex_hits = [term for term in VIET_TARGET_LEXICON if term in seg.lower()]
    combined = list(set(entities + lex_hits))
    print(f"  {t!r}")
    print(f"    NER: {entities if entities else '[]'}")
    print(f"    Lex: {lex_hits[:5]}{'...' if len(lex_hits) > 5 else ''}")
    print(f"    Combined: {combined if combined else '[CLS fallback]'}")

## 9. AmpleHate Dataset and DataLoader

The AmpleHate dataset extends the standard text dataset with `head_token_idx`:
a list of token positions for NER-detected entities (or [0] as CLS fallback).

During training: NER is applied to find explicit targets.
During validation/test: NER is disabled (use_ner=False) → always CLS fallback.
This matches the original AmpleHate implementation exactly.

`collate_fn` pads `head_token_idx` to the maximum number of entities in the batch
(zero-padding = CLS position, which is safe for the model).

In [ ]:
class AmpleHateDataset(Dataset):
    """AmpleHate dataset with Vietnamese improvements (Improvements 2, 3).
    Stores both raw_texts (for NER) and texts (segmented, for PhoBERT).
    """
    def __init__(self, df, ner_processor):
        self.raw_texts = df['raw_text'].fillna('').tolist()       # unsegmented — for NER
        self.texts     = df['text_processed'].fillna('').tolist() # segmented — for PhoBERT
        self.labels    = df['label_id'].astype(int).tolist()
        self.processor = ner_processor

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        token_ids, head_token_idx, attention_mask = self.processor.tokenize_and_encode(
            self.texts[idx], self.raw_texts[idx]  # pass both segmented and raw
        )
        return {
            'input_ids':      torch.tensor(token_ids,      dtype=torch.long),
            'head_token_idx': torch.tensor(head_token_idx, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }


def collate_fn(batch):
    """Pads head_token_idx to max entity count in the batch."""
    max_heads = max(len(item['head_token_idx']) for item in batch)
    padded_heads = []
    for item in batch:
        h = item['head_token_idx']
        pad = torch.zeros(max_heads - len(h), dtype=torch.long)
        padded_heads.append(torch.cat([h, pad]))
    return {
        'input_ids':      torch.stack([b['input_ids']      for b in batch]),
        'head_token_idx': torch.stack(padded_heads),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'labels':         torch.stack([b['labels']         for b in batch]),
    }

In [ ]:
print("Building datasets...")
print("  Train: applying Vietnamese NER + lexicon...")
train_ds = AmpleHateDataset(train_df, ner_processor_train)
print("  Val/Test: NER disabled (lexicon only via CLS fallback path)...")
val_ds   = AmpleHateDataset(val_df,   ner_processor_eval)
test_ds  = AmpleHateDataset(test_df,  ner_processor_eval)

g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, generator=g,
    num_workers=0, pin_memory=PIN_MEMORY  # NER pipeline is not fork-safe
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)

print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

In [ ]:
def ner_coverage_statistics(dataset, name="dataset"):
    total = len(dataset)
    ner_applied = 0
    print(f"Computing NER coverage for {name} ({total} samples)...")
    for i in range(total):
        item = dataset[i]
        # NER applied if head_token_idx is NOT just [0] (i.e., not pure CLS fallback)
        if not (len(item['head_token_idx']) == 1 and item['head_token_idx'][0].item() == 0):
            ner_applied += 1
    print(f"  NER entities found in: {ner_applied}/{total} ({ner_applied/total*100:.2f}%)")
    print(f"  CLS fallback used for: {total-ner_applied}/{total} ({(total-ner_applied)/total*100:.2f}%)")

# NOTE: This call runs NER on all ~11k training samples (~45 min on T4).
# Skip this cell or move it to after training to avoid doubling NER overhead.
ner_coverage_statistics(train_ds, "Train")

## 10. AmpleHate Model: HeadAttention + PhoBERT

Direct port of the original AmpleHate model (`model/model.py`) with PhoBERT
replacing BERT-base-uncased. Architecture unchanged:

1. **HeadAttention**: Q,V from [CLS]; K from target entity token.
   `score = softmax(W_q·CLS · (W_k·target)^T / sqrt(d))`
   `output = score · W_v·CLS`

2. **Forward pass**:
   ```
   cls = PhoBERT([CLS] token embedding)
   for each entity token at head_token_idx:
       head_attn += HeadAttention(cls, entity_token)
   final = cls + e * head_attn
   logits = Linear(Dropout(final))
   ```
   
When all head_token_idx = 0 (CLS fallback), the model still differs from
plain PhoBERT because HeadAttention applies learned W_q, W_k, W_v projections.

In [ ]:
class HeadAttention(nn.Module):
    """Original AmpleHate HeadAttention (model/model.py:5-27, unchanged)."""
    def __init__(self, hidden_dim, head_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.head_dim   = head_dim
        self.softmax    = nn.Softmax(dim=-1)
        self.W_q = nn.Linear(hidden_dim, head_dim, bias=False)
        self.W_k = nn.Linear(hidden_dim, head_dim, bias=False)
        self.W_v = nn.Linear(hidden_dim, head_dim, bias=False)

    def forward(self, cls_embedding, head_token_embedding):
        Q_h = self.W_q(cls_embedding)          # [batch, head_dim]
        K_h = self.W_k(head_token_embedding)   # [batch, head_dim]
        V_h = self.W_v(cls_embedding)          # [batch, head_dim]
        scores  = torch.matmul(Q_h, K_h.T) / (self.head_dim ** 0.5)
        scores  = scores.float()
        weights = self.softmax(scores)
        return torch.matmul(weights, V_h)      # [batch, head_dim]


class AmpleHatePhoBERT(nn.Module):
    """ViAmpleHate: AmpleHate with PhoBERT encoder and contrastive loss support."""
    def __init__(self, model_name, hidden_dim=HIDDEN_DIM, e=E_INJECTION, dropout=DROPOUT):
        super().__init__()
        self.bert           = AutoModel.from_pretrained(model_name)
        self.hidden_dim     = hidden_dim
        self.e              = e
        self.head_attention = HeadAttention(hidden_dim, HEAD_DIM)
        self.dropout        = nn.Dropout(dropout)
        self.classifier     = nn.Linear(hidden_dim, NUM_CLASSES)
        self.last_embedding = None  # set in forward; used by ContrastiveLoss (Improvement 5)

    def forward(self, input_ids, head_token_idx, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # [batch, hidden]

        expanded_idx = head_token_idx.unsqueeze(-1).expand(-1, -1, self.hidden_dim)
        head_token_embeddings = torch.gather(outputs.last_hidden_state, 1, expanded_idx)

        outputs_list = [
            self.head_attention(cls_embedding, head_token_embeddings[:, i, :])
            for i in range(head_token_embeddings.shape[1])
        ]
        head_attention_output = sum(outputs_list)

        # Direct injection: cls + e * attention_output (AmpleHate core contribution)
        final_embedding = cls_embedding + head_attention_output * self.e
        self.last_embedding = final_embedding.detach()  # Improvement 5: for ContrastiveLoss
        final_embedding = self.dropout(final_embedding)

        return self.classifier(final_embedding)

In [ ]:
class ContrastiveLossCosine(nn.Module):
    """Original AmpleHate contrastive loss (model/cl_loss.py, unchanged)."""
    def __init__(self, margin=0.5):
        super().__init__()
        self.margin = margin

    def forward(self, embeddings, labels):
        # NOTE: requires batch_size >= 2; denominator is batch*(batch-1)
        batch_size  = embeddings.size(0)
        cosine_sim  = F.cosine_similarity(
            embeddings.unsqueeze(1), embeddings.unsqueeze(0), dim=-1
        )  # [batch, batch]
        labels      = labels.unsqueeze(1)
        label_matrix = (labels != labels.T).float()
        pos_loss    = (1 - label_matrix) * (1 - cosine_sim)
        neg_loss    = label_matrix * F.relu(cosine_sim - self.margin)
        return (pos_loss + neg_loss).sum() / (batch_size * (batch_size - 1))

In [ ]:
print('Loading AmpleHate + PhoBERT model...')
model = AmpleHatePhoBERT(MODEL_NAME, hidden_dim=HIDDEN_DIM, e=E_INJECTION, dropout=DROPOUT).to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total:,}')
print(f'Trainable params: {trainable:,}')
print(f'Encoder         : {MODEL_NAME}')
print(f'e (injection)   : {E_INJECTION}')
print(f'Dropout         : {DROPOUT}')

## 11. Loss, Optimizer, and Scheduler

Following the original AmpleHate training setup:
- **Loss**: CrossEntropy (primary). The original also supports contrastive loss
  but we use CE-only for this baseline.
- **Optimizer**: AdamW, lr=2e-5 (original default).
- **Class weights**: added to handle ViHSD class imbalance (~89% NON-HATE).
- **Scheduler**: linear warmup added (not in original, but standard for PhoBERT).
- **Differential LR**: higher LR for HeadAttention + classifier head (not in original).

In [ ]:
label_counts  = train_df['label_id'].value_counts().sort_index().values
class_weights = torch.tensor(
    len(train_df) / (NUM_CLASSES * label_counts),
    dtype=torch.float32, device=DEVICE
)
print('Class weights:', class_weights.cpu().numpy().round(3))

criterion_train = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
criterion_eval  = nn.CrossEntropyLoss()
criterion_cl    = ContrastiveLossCosine(margin=CONTRASTIVE_MARGIN)  # Improvement 5

no_decay    = ['bias', 'LayerNorm.weight']
bert_decay, bert_no_decay = [], []
for name, param in model.bert.named_parameters():
    if not param.requires_grad:
        continue
    (bert_no_decay if any(nd in name for nd in no_decay) else bert_decay).append(param)

head_params = (
    list(model.head_attention.parameters()) +
    list(model.classifier.parameters())
)

optimizer = optim.AdamW([
    {'params': bert_decay,    'lr': LR,      'weight_decay': WEIGHT_DECAY},
    {'params': bert_no_decay, 'lr': LR,      'weight_decay': 0.0},
    {'params': head_params,   'lr': HEAD_LR, 'weight_decay': WEIGHT_DECAY},
])

total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler    = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')
print(f'Total steps: {total_steps} | Warmup: {warmup_steps}')
print(f'ContrastiveLoss: margin={CONTRASTIVE_MARGIN}, lambda={LAMBDA_CL}')

## 12. Training and Evaluation Loops

Training loop follows the original AmpleHate `train_epoch` and `evaluate` functions.
Key differences from original:
- AMP (mixed precision) added for Kaggle T4 efficiency.
- Gradient clipping added (norm=1.0) for stability.
- Best threshold search on validation set (from original `best_threshold` function).

In [ ]:
def best_threshold(probs: np.ndarray, labels: np.ndarray,
                   grid=np.linspace(0.05, 0.95, 19)) -> float:
    """Grid-search for best classification threshold on validation set.
    Direct copy of train.py:best_threshold (unchanged).
    """
    best_t, best_f1 = 0.5, 0.0
    for t in grid:
        f1 = f1_score(labels, (probs >= t).astype(int), average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t)

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion, criterion_cl, scaler):
    model.train()
    total_loss = total_correct = total_n = 0

    for batch in loader:
        ids   = batch['input_ids'].to(DEVICE,      non_blocking=PIN_MEMORY)
        heads = batch['head_token_idx'].to(DEVICE,  non_blocking=PIN_MEMORY)
        mask  = batch['attention_mask'].to(DEVICE,  non_blocking=PIN_MEMORY)
        y     = batch['labels'].to(DEVICE,          non_blocking=PIN_MEMORY)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
            logits  = model(ids, heads, mask)
            ce_loss = criterion(logits, y)
            cl_loss = criterion_cl(model.last_embedding, y)  # Improvement 5
            loss    = ce_loss + LAMBDA_CL * cl_loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        preds = logits.argmax(1)
        total_loss    += loss.item() * y.size(0)
        total_correct += (preds == y).sum().item()
        total_n       += y.size(0)

    return total_loss / total_n, total_correct / total_n

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = total_n = 0
    all_probs, all_labels = [], []

    for batch in loader:
        ids   = batch['input_ids'].to(DEVICE,      non_blocking=PIN_MEMORY)
        heads = batch['head_token_idx'].to(DEVICE,  non_blocking=PIN_MEMORY)
        mask  = batch['attention_mask'].to(DEVICE,  non_blocking=PIN_MEMORY)
        y     = batch['labels'].to(DEVICE,          non_blocking=PIN_MEMORY)

        with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
            logits = model(ids, heads, mask)
            loss   = criterion(logits, y)

        probs = torch.softmax(logits, dim=1)[:, 1]

        total_loss    += loss.item() * y.size(0)
        total_n       += y.size(0)

        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)

    t      = best_threshold(all_probs, all_labels)
    y_pred = (all_probs >= t).astype(int)
    acc    = accuracy_score(all_labels, y_pred)
    macro_f1 = f1_score(all_labels, y_pred, average='macro', zero_division=0)

    return total_loss / total_n, acc, macro_f1, float(t)

In [ ]:
history = {
    'train_loss': [], 'val_loss': [],
    'train_acc':  [], 'val_acc':  [],
    'val_f1':     [], 'threshold': []
}
best_f1, best_epoch, best_t_saved, patience_counter = -1.0, 0, 0.5, 0

hdr = f"{'Epoch':>6} | {'Tr Loss':>8} | {'Tr Acc':>7} | {'Val Loss':>8} | {'Val Acc':>7} | {'Val F1':>6} | {'Thresh':>6} | {'LR':>8} | {'Time':>6}"
print(hdr)
print('-' * len(hdr))

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_epoch(
        model, train_loader, optimizer, scheduler, criterion_train, criterion_cl, scaler
    )
    vl_loss, vl_acc, vl_f1, vl_t = evaluate(model, val_loader, criterion_eval)
    elapsed = time.time() - t0
    current_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)
    history['val_f1'].append(vl_f1)
    history['threshold'].append(vl_t)

    flag = ''
    if vl_f1 > best_f1:
        best_f1, best_epoch, best_t_saved = vl_f1, epoch, vl_t
        torch.save({'model': model.state_dict(), 'threshold': vl_t}, CKPT_NAME)
        patience_counter = 0
        flag = ' *saved*'
    else:
        patience_counter += 1

    print(
        f'{epoch:>6} | {tr_loss:>8.4f} | {tr_acc:>7.4f} | '
        f'{vl_loss:>8.4f} | {vl_acc:>7.4f} | {vl_f1:>6.4f} | '
        f'{vl_t:>6.2f} | {current_lr:>8.2e} | {elapsed:>5.1f}s{flag}'
    )

    if patience_counter >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'\nBest Val Macro-F1 = {best_f1:.4f} at epoch {best_epoch} (threshold={best_t_saved:.2f})')

In [ ]:
# Improvement 6: e-injection sweep (COMMENTED OUT — activates 5x training time)
# Run this after Improvements 1-3 are confirmed active (NER coverage > 20%).
# Higher e is better when NER hits are reliable; lower e reduces noise from CLS-on-CLS fallback.
#
# for e_val in [0.5, 0.75, 1.0, 1.25, 1.5]:
#     print(f'\n--- e = {e_val} ---')
#     m = AmpleHatePhoBERT(MODEL_NAME, hidden_dim=HIDDEN_DIM, e=e_val, dropout=DROPOUT).to(DEVICE)
#     no_d = ['bias', 'LayerNorm.weight']
#     bd, bnd = [], []
#     for n, p in m.bert.named_parameters():
#         if not p.requires_grad: continue
#         (bnd if any(nd in n for nd in no_d) else bd).append(p)
#     hp = list(m.head_attention.parameters()) + list(m.classifier.parameters())
#     opt = optim.AdamW([
#         {'params': bd,  'lr': LR,      'weight_decay': WEIGHT_DECAY},
#         {'params': bnd, 'lr': LR,      'weight_decay': 0.0},
#         {'params': hp,  'lr': HEAD_LR, 'weight_decay': WEIGHT_DECAY},
#     ])
#     ts = len(train_loader) * NUM_EPOCHS
#     ws = int(ts * WARMUP_RATIO)
#     sch = get_linear_schedule_with_warmup(opt, ws, ts)
#     sc  = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')
#     cl  = ContrastiveLossCosine(margin=CONTRASTIVE_MARGIN)
#     best_e_f1 = -1.0
#     for ep in range(1, NUM_EPOCHS + 1):
#         train_epoch(m, train_loader, opt, sch, criterion_train, cl, sc)
#         _, _, f1, t = evaluate(m, val_loader, criterion_eval)
#         if f1 > best_e_f1:
#             best_e_f1 = f1
#         print(f'  Epoch {ep}: Val F1={f1:.4f}, Threshold={t:.2f}')
#     print(f'  => Best Val F1 for e={e_val}: {best_e_f1:.4f}')

## 13. Training Curves

In [ ]:
n            = len(history['train_loss'])
epochs_range = range(1, n + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train')
axes[0].plot(epochs_range, history['val_loss'],   'r-o', label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(epochs_range, history['train_acc'], 'b-o', label='Train')
axes[1].plot(epochs_range, history['val_acc'],   'r-o', label='Val')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].set_xlabel('Epoch')

axes[2].plot(epochs_range, history['val_f1'], 'g-o')
axes[2].axvline(best_epoch, color='red', linestyle='--', label=f'Best (epoch {best_epoch})')
axes[2].set_title('Val Macro F1'); axes[2].legend(); axes[2].set_xlabel('Epoch')

plt.suptitle(PLOT_TITLE, fontsize=14)
plt.tight_layout()
plt.savefig('training_curves_amplehate.png', dpi=150)
plt.show()

## 14. Test Set Evaluation

Loading best checkpoint and evaluating on the held-out test set.
Threshold from validation grid search is applied.

In [ ]:
ckpt = torch.load(CKPT_NAME, map_location=DEVICE)
model.load_state_dict(ckpt['model'])
threshold = ckpt['threshold']
print(f'Loaded checkpoint from epoch {best_epoch}, threshold={threshold:.2f}')

In [ ]:
@torch.no_grad()
def get_predictions(model, loader, threshold):
    model.eval()
    all_probs, all_labels = [], []

    for batch in loader:
        ids   = batch['input_ids'].to(DEVICE,      non_blocking=PIN_MEMORY)
        heads = batch['head_token_idx'].to(DEVICE,  non_blocking=PIN_MEMORY)
        mask  = batch['attention_mask'].to(DEVICE,  non_blocking=PIN_MEMORY)

        with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
            logits = model(ids, heads, mask)
        probs = torch.softmax(logits, dim=1)[:, 1]

        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(batch['labels'].numpy())

    y_pred = (np.array(all_probs) >= threshold).astype(int)
    return np.array(all_labels), y_pred

y_true, y_pred = get_predictions(model, test_loader, threshold)
print('Classification Report — Test Set')
print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=4))

## 15. Full Test Metrics

In [ ]:
acc      = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro',    zero_division=0)
macro_p  = precision_score(y_true, y_pred, average='macro', zero_division=0)
macro_r  = recall_score(y_true, y_pred, average='macro',    zero_division=0)
hate_f1  = f1_score(y_true, y_pred, labels=[1], average='macro', zero_division=0)

print(f"Accuracy           : {acc:.4f}")
print(f"Macro Precision    : {macro_p:.4f}")
print(f"Macro Recall       : {macro_r:.4f}")
print(f"Macro F1           : {macro_f1:.4f}")
print(f"F1 (HATE class)    : {hate_f1:.4f}")

In [ ]:
cm     = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(cm,     annot=True, fmt='d',   cmap='Blues',   ax=axes[0],
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Oranges', ax=axes[1],
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
axes[0].set_title('Confusion Matrix (count)')
axes[1].set_title('Confusion Matrix (% per row)')
for ax in axes:
    ax.set_ylabel('True'); ax.set_xlabel('Predicted')
plt.suptitle(PLOT_TITLE + ' — Test Set', fontsize=14)
plt.tight_layout()
plt.savefig('confusion_matrix_amplehate.png', dpi=150)
plt.show()

## 16. Quick Inference

Run a few hand-written examples through the trained AmpleHate PhoBERT model.
The prediction uses the validation threshold saved in the best checkpoint.

In [ ]:
@torch.no_grad()
def predict(texts):
    model.eval()
    results = []

    for text in texts:
        processed = preprocess(text)
        token_ids, head_token_idx, attention_mask = ner_processor_eval.tokenize_and_encode(processed)

        ids = torch.tensor([token_ids], dtype=torch.long, device=DEVICE)
        heads = torch.tensor([head_token_idx], dtype=torch.long, device=DEVICE)
        mask = torch.tensor([attention_mask], dtype=torch.long, device=DEVICE)

        with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
            logits = model(ids, heads, mask)

        probs = torch.softmax(logits, dim=-1).squeeze(0).cpu().numpy()
        hate_prob = float(probs[1])
        pred_id = int(hate_prob >= threshold)

        results.append({
            'text': text,
            'processed': processed,
            'label': LABEL_NAMES[pred_id],
            'threshold': round(float(threshold), 2),
            'scores': {LABEL_NAMES[i]: round(float(p), 4) for i, p in enumerate(probs)},
        })

    return results


TEST_CASES = [
    ('Hôm nay trời đẹp quá, đi chơi thôi!', 'NON-HATE'),
    ('Tụi nó toàn nói nhảm, đúng là quá toxic', 'NON-HATE'),
    ('Đồ ngu, câm miệng lại đi mày', 'NON-HATE'),
    ('Tao ghét cái loại người như mày, xéo đi cho khuất mắt', 'HATE'),
    ('Cảm ơn bạn đã giúp đỡ mình nhé!', 'NON-HATE'),
    ('Tao cảm ơn mày nhiều lắm', 'NON-HATE'),
    ('nguyên cả cái tỉnh này không được khôn lắm', 'HATE'),
]

correct = 0
for text, expected in TEST_CASES:
    r = predict([text])[0]
    match = 'OK' if r['label'] == expected else 'WRONG'
    correct += int(r['label'] == expected)
    prob_str = ' | '.join([f'{k}: {v:.2f}' for k, v in r['scores'].items()])

    print(f'Input    : {text}')
    print(f'Processed: {r["processed"]}')
    print(f'Expected : {expected}')
    print(f'Pred     : {r["label"]:<10} {match:<6} {prob_str} | threshold={r["threshold"]:.2f}')
    print()

print(f'Summary: {correct}/{len(TEST_CASES)}  ({correct / len(TEST_CASES) * 100:.0f}%)')

In [ ]:
os.makedirs('outputs', exist_ok=True)

config = {
    'notebook'           : 'vihsd-viamplehate-phobert-proposed',
    'dataset'            : 'ViHSD (sonlam1102/vihsd)',
    'method'             : 'ViAmpleHate (Vietnamese NER + Lexicon + ContrastiveLoss)',
    'encoder'            : MODEL_NAME,
    'ner_model'          : NER_MODEL,
    'max_len'            : MAX_LEN,
    'hidden_dim'         : HIDDEN_DIM,
    'e_injection'        : E_INJECTION,
    'lambda_cl'          : LAMBDA_CL,
    'contrastive_margin' : CONTRASTIVE_MARGIN,
    'dropout'            : DROPOUT,
    'num_classes'        : NUM_CLASSES,
    'label_names'        : LABEL_NAMES,
    'lr_encoder'         : float(LR),
    'lr_head'            : float(HEAD_LR),
    'best_epoch'         : int(best_epoch),
    'best_val_f1'        : round(float(best_f1), 4),
    'best_threshold'     : round(float(threshold), 2),
    'test_accuracy'      : round(float(acc), 4),
    'test_macro_f1'      : round(float(macro_f1), 4),
    'test_macro_p'       : round(float(macro_p), 4),
    'test_macro_r'       : round(float(macro_r), 4),
    'test_f1_hate'       : round(float(hate_f1), 4),
}

with open('outputs/viamplehate_vihsd_config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(json.dumps(config, indent=2, ensure_ascii=False))

## 17. What Was Improved and What Remains

This notebook applies the highest-priority improvements from `docs/improvementAmpleHate.md`.

---

### Applied Improvements

| # | Improvement | Status |
|---|---|---|
| 1 | Vietnamese NER (`NlpHUST/ner-vietnamese-electra-base`) | Active |
| 2 | Vietnamese hate-target lexicon | Active |
| 3 | Word segmentation alignment (NER on raw text) | Active |
| 4 | Vietnamese target types (VLSP PER/ORG/LOC/MISC) | Active (via Imp 1) |
| 5 | ContrastiveLossCosine (CE + lambda*CL) | Active |

---

### Optional / Disabled

| # | Improvement | Status |
|---|---|---|
| 6 | e-injection sweep [0.5, 0.75, 1.0, 1.25, 1.5] | Commented cell after training loop |
| 7 | max_length=256 (if truncation > 5%) | Profiling cell included; MAX_LEN=128 default |
| 8 | PhoBERT-large (HIDDEN_DIM=1024, BATCH_SIZE=8) | USE_PHOBERT_LARGE=False flag in Sec 2 |

---

### Not Applied (out of scope)

| # | Improvement | Reason |
|---|---|---|
| 9 | Within-example HeadAttention | Deviates from original AmpleHate; reserved for variant comparison |
| 10 | Multi-class 3-label setup | Optional/advanced; requires separate ablation |

---

### Summary: What Makes ViAmpleHate Different

| Component | Baseline | Proposed (this notebook) |
|---|---|---|
| NER model | English CoNLL-2003 | Vietnamese VLSP (NlpHUST ELECTRA) |
| Target coverage | ~0.09% | ~20-40% (NER + lexicon) |
| Lexicon | None | 200+ Vietnamese hate-target terms |
| Segmentation | Mismatch (NER on segmented) | Fixed (NER on raw, mapped to segmented positions) |
| Loss | CrossEntropy only | CE + 0.1 * ContrastiveLoss |